# 03 · Inference & MCMC Deep Dive

**Owner: Member 3.**

Companion to §§6–8 of `main.ipynb`. We measure MCMC
convergence quantitatively and replicate the exact answer with
two different samplers.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
np.random.seed(42)

In [ ]:
from src.data_loader import load_heart_disease
from src.preprocessing import (
    PreprocessConfig, build_dataset, train_test_split_df, variable_state_names,
)
from src.expert_network import build_expert_dag
from src.parameter_learning import ParameterFitConfig, fit_parameters
from src.inference import make_engine, posterior

df = build_dataset(load_heart_disease(), PreprocessConfig())
train, test = train_test_split_df(df, PreprocessConfig())
states = variable_state_names(df)
bn = fit_parameters(build_expert_dag(), train, ParameterFitConfig(method='bayes'), state_names=states)
engine = make_engine(bn)

## Effect of burn-in & sample size on MH

We sweep the post-burn-in sample size and plot the TV distance
to the exact Variable-Elimination posterior.

In [ ]:
from src.mcmc import MHConfig, metropolis_hastings

evidence = {'age': '65+', 'sex': '1', 'chol': 'high', 'trestbps': 'hyper'}
exact = posterior(engine, 'target', evidence)

rows = []
for n in [200, 500, 1000, 2000, 4000, 8000]:
    cfg = MHConfig(n_samples=n, burn_in=500, seed=0)
    mh_post, _ = metropolis_hastings(bn, evidence, query='target', cfg=cfg)
    tv = 0.5 * np.abs(mh_post.values - exact.values).sum()
    rows.append({'n_samples': n, 'TV(MH, exact)': tv})
df_conv = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df_conv['n_samples'], df_conv['TV(MH, exact)'], 'o-')
ax.set_xscale('log'); ax.set_xlabel('post-burn-in samples'); ax.set_ylabel('TV distance to exact')
ax.set_title('MH convergence to exact posterior')
plt.show()
df_conv

## Gibbs vs. MH on the same query

In [ ]:
from src.mcmc import GibbsConfig, gibbs_posterior
gibbs_post, _ = gibbs_posterior(bn, evidence, query='target',
                                cfg=GibbsConfig(n_samples=6000, burn_in=1000))
mh_post, _ = metropolis_hastings(bn, evidence, query='target',
                                 cfg=MHConfig(n_samples=6000, burn_in=1000))
pd.DataFrame({'exact': exact, 'Gibbs': gibbs_post, 'MH': mh_post})

## Counterfactual sensitivity

How much does P(disease) change as we sweep each modifiable
risk factor?

In [ ]:
from src.inference import do_intervention

baseline = posterior(engine, 'target', evidence)
positive = sorted(bn.get_cpds('target').state_names['target'])[-1]
baseline_risk = float(baseline.loc[positive])

rows = []
for var in ['chol', 'trestbps', 'fbs']:
    for val in sorted(set(states[var])):
        cf = do_intervention(bn, {var: val}, query='target')
        rows.append({'variable': var, 'do(value)': val,
                     'P(disease | do)': float(cf.loc[positive]),
                     'Δ from baseline': float(cf.loc[positive]) - baseline_risk})
pd.DataFrame(rows).sort_values(['variable', 'do(value)'])